In [0]:
%sql

create schema if not exists healthcare_proj_databricks_ws.silver;

In [0]:
bronze_table = 'healthcare_proj_databricks_ws.bronze.visit_raw'
silver_table = 'healthcare_proj_databricks_ws.silver.fact_visit'
checkpoint_path = "abfss://data@healthcaredbprojdl.dfs.core.windows.net/silver/fact_visit/checkpoint/"

In [0]:

from pyspark.sql.functions import col, lag, to_date, datediff, current_timestamp
from delta.tables import DeltaTable

In [0]:
%sql

select * from healthcare_proj_databricks_ws.bronze.visit_raw;

In [0]:
silver_patient_table = "healthcare_proj_databricks_ws.silver.dim_patient"
silver_hospital_table = "healthcare_proj_databricks_ws.silver.dim_hospital"
silver_diagnosis_table = "healthcare_proj_databricks_ws.silver.dim_diagnosis"
bronze_table = "healthcare_proj_databricks_ws.bronze.visit_raw"

In [0]:

df_patient = spark.read.table(silver_patient_table)
df_hospital = spark.read.table(silver_hospital_table)
df_diagnosis = spark.read.table(silver_diagnosis_table)

In [0]:
df_visit_bronze = (
    spark.readStream.table(bronze_table)
)

In [0]:
# Rename columns to avoid duplicates
df_patient = df_patient.withColumnRenamed("city", "patient_city")

df_hospital = df_hospital.withColumnRenamed("city", "hospital_city")


# Join clean fact visit with dimension tables
df_fact_combined = (
    df_visit_bronze
        .join(df_patient, "patient_id", "left")
        .join(df_hospital, "hospital_id", "left")
        .join(df_diagnosis, "diagnosis_code", "left")
        .withColumn("admission_date", to_date("admission_date"))
        .withColumn("discharge_date", to_date("discharge_date"))
        .withColumn("load_timestamp", current_timestamp())
)

In [0]:
# -------------------------
# Merge into Silver fact_visit
# -------------------------
def merge_fact_visit(batch_df, batch_id):

    if not spark.catalog.tableExists(silver_table):
        batch_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
        return

    fact = DeltaTable.forName(spark, silver_table)
    fact.alias("t").merge(
        batch_df.alias("s"),
        "t.visit_id = s.visit_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

In [0]:
# -------------------------
# Run as availableNow incremental
# -------------------------
query = (
    df_fact_combined.drop("load_timestamp").writeStream
        .foreachBatch(merge_fact_visit)
        .outputMode("update")
        .trigger(availableNow=True)
        .option("checkpointLocation", checkpoint_path)
        .start()
)

query.awaitTermination()  # Block until the stream finishes before proceeding

In [0]:
%sql

select * from healthcare_proj_databricks_ws.silver.fact_visit;